In [1]:
import json
import pandas as pd

# Load your JSON file
with open("nvd_dataset.json", "r",  encoding="utf-8") as f:
    data = json.load(f)

# Extract key info
records = []
for item in data["CVE_Items"]:
    cve_id = item["cve"]["CVE_data_meta"]["ID"]
    description = item["cve"]["description"]["description_data"][0]["value"]
    published = item["publishedDate"]
    impact = item.get("impact", {})
    
    # CVSS score if available
    base_score = None
    if "baseMetricV3" in impact:
        base_score = impact["baseMetricV3"]["cvssV3"]["baseScore"]
    elif "baseMetricV2" in impact:
        base_score = impact["baseMetricV2"]["cvssV2"]["baseScore"]

    records.append({
        "cve_id": cve_id,
        "description": description,
        "published": published,
        "cvss_score": base_score
    })

df = pd.DataFrame(records)
print(df.head())


          cve_id                                        description  \
0  CVE-2025-0001  Abacus ERP is versions older than 2024.210.160...   
1  CVE-2025-0014  Incorrect default permissions on the AMD Ryzen...   
2  CVE-2025-0015  Use After Free vulnerability in Arm Ltd Valhal...   
3  CVE-2025-0050  Improper Restriction of Operations within the ...   
4  CVE-2025-0053  SAP NetWeaver Application Server for ABAP and ...   

           published  cvss_score  
0  2025-02-17T10:15Z         NaN  
1  2025-04-02T17:15Z         NaN  
2  2025-02-03T11:15Z         NaN  
3  2025-04-07T12:15Z         NaN  
4  2025-01-14T01:15Z         NaN  


In [2]:
def score_to_severity(score):
    if score is None:
        return "Unknown"
    elif score < 4.0:
        return "Low"
    elif score < 7.0:
        return "Medium"
    else:
        return "High"

df["severity"] = df["cvss_score"].apply(score_to_severity)
df = df[df["severity"] != "Unknown"]  # Filter out unknown labels


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(df["description"], df["severity"], test_size=0.2, random_state=42)

# TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

        High       0.90      0.98      0.94      1419
         Low       1.00      0.08      0.14        26
      Medium       0.78      0.46      0.58       257

    accuracy                           0.89      1702
   macro avg       0.89      0.51      0.55      1702
weighted avg       0.88      0.89      0.87      1702



In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Encode severity labels to integers
le = LabelEncoder()
y_encoded = le.fit_transform(df["severity"])

# Tokenize text
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["description"])
sequences = tokenizer.texts_to_sequences(df["description"])

max_len = 100
padded = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(padded, y_encoded, test_size=0.2, random_state=42)

# Build BiLSTM Model
model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))


D:\Anaconda\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 25s 86ms/step - accuracy: 0.8195 - loss: 0.5406 - val_accuracy: 0.8749 - val_loss: 0.3260
Epoch 2/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 17s 79ms/step - accuracy: 0.8985 - loss: 0.2698 - val_accuracy: 0.8766 - val_loss: 0.3081
Epoch 3/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 17s 79ms/step - accuracy: 0.9261 - loss: 0.1906 - val_accuracy: 0.8860 - val_loss: 0.3017
Epoch 4/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - accuracy: 0.9514 - loss: 0.1351 - val_accuracy: 0.8760 - val_loss: 0.4169
Epoch 5/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 17s 78ms/step - accuracy: 0.9615 - loss: 0.1126 - val_accuracy: 0.8895 - val_loss: 0.3718
Epoch 6/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 17s 79ms/step - accuracy: 0.9657 - loss: 0.0887 - val_accuracy: 0.8890 - val_loss: 0.4077
Epoch 7/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 17s 79ms/step - accuracy: 0.9733 - loss: 0.0757 - val_accuracy: 0.8684 - val_loss: 0.4310
Epoch 8/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - accuracy: 0.9761 - loss: 0.0640 - 

In [5]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000), params, cv=5)
grid.fit(X_train_tfidf, y_train)
print(grid.best_params_)


{'C': 10, 'solver': 'liblinear'}


In [10]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
xgb.fit(X_train_tfidf, y_train)
print(classification_report(y_test, xgb.predict(X_test_tfidf)))


D:\Anaconda\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:34:54] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       0.92      0.97      0.94      1419
           1       0.57      0.15      0.24        26
           2       0.73      0.54      0.62       257

    accuracy                           0.89      1702
   macro avg       0.74      0.56      0.60      1702
weighted avg       0.88      0.89      0.88      1702



In [14]:
from sklearn.metrics import confusion_matrix, classification_report

# Convert both y_test and y_pred to string labels
y_test_str = le.inverse_transform(y_test)
y_pred_xgb = le.inverse_transform(xgb.predict(X_test_tfidf))

# Print confusion matrix and classification report
print(confusion_matrix(y_test_str, y_pred_xgb))
print(classification_report(y_test_str, y_pred_xgb))


[[1378    1   40]
 [  10    4   12]
 [ 116    2  139]]
              precision    recall  f1-score   support

        High       0.92      0.97      0.94      1419
         Low       0.57      0.15      0.24        26
      Medium       0.73      0.54      0.62       257

    accuracy                           0.89      1702
   macro avg       0.74      0.56      0.60      1702
weighted avg       0.88      0.89      0.88      1702

